In [83]:
import pandas as pd                                      # tables (dataframes)
from mlxtend.preprocessing import TransactionEncoder    # one hot encoding: baskets > T/F table
from mlxtend.frequent_patterns import apriori           # find the frequent itemsets
from mlxtend.frequent_patterns import association_rules  # find the association rules
from icecream import ic

In [84]:
'''
T1 -> bread, tea represented by 
T1 -> bread
T1 -> tea

T2 -> sugar, bulb represented by 
T2 -> sugar
T2 -> bulb

T3-> tea, sugar
...

Item = {bread, tea, sugar, bulb}

TransactionEncoder (blank means false for this representation)
    bread, tea, sugar, bulb
T1    T     T
T2                T      T
T3          T     T      
'''

'\nT1 -> bread, tea represented by \nT1 -> bread\nT1 -> tea\n\nT2 -> sugar, bulb represented by \nT2 -> sugar\nT2 -> bulb\n\nT3-> tea, sugar\n...\n\nItem = {bread, tea, sugar, bulb}\n\nTransactionEncoder (blank means false for this representation)\n    bread, tea, sugar, bulb\nT1    T     T\nT2                T      T\nT3          T     T      \n'

# Step 1 - Read the transaction file

In [85]:
'''
The file has ONE ROW PER LINE ITEM (not one row per backet).
Several rows share the same transaction_id: those rows together are one receipt. 
'''

file_path = 'grocery_transactions.csv'
df = pd.read_csv(file_path)

print('STEP 1 - File Loaded.')
print(f'     rows (line items): {len(df)}')
print(f'     transactions:      {len(df["transaction_id"].unique())}')
print(f'     columns:           {list(df.columns)}') # list gives better formatting

STEP 1 - File Loaded.
     rows (line items): 13234
     transactions:      2472
     columns:           ['transaction_id', 'timestamp', 'customer_id', 'item', 'category', 'quantity', 'unit_price', 'line_total']


# Step 2 - Organize the data: one list of items per transaction

In [86]:
''' 
2a) Clean the product names first
- remove spaces at the start and the end ("    butter " -> "butter")
- put everything in lower case ("Butter" -> "butter")
without this, "Butter" and "butter" will be treated as different items, and "    butter " will be treated as a different item from "butter"
'''
df['item'] = df['item'].str.strip().str.lower()

''' 
2b) Group the rows by receipts (transaction_id), and collect the items into a list for each transaction.
- gather the item values of each transaction
- run a small function on each group
- remove the duplicates (if any) in each basket (transaction)
- (an itme scanned twice is still the same item)
- put them in a fixed order (sorted) so that the same basket is always represented the same way and easier to read
'''
baskets = df.groupby('transaction_id')['item'].apply(lambda items: sorted(set(items))
            ) # lambda = a one-line function that takes a list of items and returns a sorted list of unique items (a set that is sorted)
print(baskets)

print('\nSTEP 2 - data organized into baskets')
print(f'     number of baskets:         {len(baskets)}')
print(f'     number of unique items:    {len(set(df["item"]))}')
# print(f'     first 3 baskets:           {baskets.head(3)}') # or baskets.iloc[0:3]
print(f'     first 3 baskets: ')
for b in baskets.head(3):
    print(f'         basket: {b}')

transaction_id
T100001                                     [eggs, parmesan]
T100002    [baby formula, beer, chicken breast, diapers, ...
T100003    [avocado, berries, granola, peanuts, whole whe...
T100004           [beer, berries, lettuce, tomatoes, yogurt]
T100005    [ground beef, orange juice, pasta, tomato sauc...
                                 ...                        
T102478    [apples, butter, conditioner, laundry detergen...
T102479    [avocado, laundry detergent, lettuce, parmesan...
T102480                      [bacon, salsa, sparkling water]
T102481    [butter, coffee beans, croissant, granola, mil...
T102482    [apples, avocado, bananas, berries, eggs, milk...
Name: item, Length: 2472, dtype: object

STEP 2 - data organized into baskets
     number of baskets:         2472
     number of unique items:    50
     first 3 baskets: 
         basket: ['eggs', 'parmesan']
         basket: ['baby formula', 'beer', 'chicken breast', 'diapers', 'red wine']
         basket: ['a

In [87]:
# Understand the difference between list, set and sorted list

my_list = {"Sugar", "Tea", "Bread", "Bulb", "Tea"}
print(my_list)
my_set = set(my_list)
print(my_set)
my_sorted_list = sorted(my_set)
print(my_sorted_list)

{'Bulb', 'Sugar', 'Tea', 'Bread'}
{'Bulb', 'Sugar', 'Tea', 'Bread'}
['Bread', 'Bulb', 'Sugar', 'Tea']


# Step 3 - Encode the baskets as a True/False Table

In [88]:
''' 
Apriori algorithm requires a one-hot encoded table (T/F table) as input (a table with one row per basket, one column per product), not a list of items.
- use TransactionEncoder to convert the list of items into a one-hot encoded table

TransactionEncoder(): the translator object
    - .fit : learn the list of all products (these become the columns of the table)
    - .transform(baskets): apply it —＞ convert the list of items into a one-hot encoded table
    - te.columns_ : the product names it learned in column order
'''

te = TransactionEncoder() 
translated_data = te.fit(baskets).transform(baskets) # We obtained a matrix of True/False
# print(translated_data, translated_data.shape)
basket_table = pd.DataFrame(translated_data, columns=te.columns_) # convert the matrix into a dataframe with the product names as column names
basket_table.to_csv('true_false_data.csv', index=False) # save the table to a csv file
# print(basket_table)

print('\nSTEP 3 - basket table built.')
print(f'    shape: {basket_table.shape}    (baskets x products)\n')
print(basket_table.iloc[:3, :5]) # print the first 3 rows and first 5 columns of the table




STEP 3 - basket table built.
    shape: (2472, 50)    (baskets x products)

   apples  avocado  baby formula  baby wipes  bacon
0   False    False         False       False  False
1   False    False          True       False  False
2   False     True         False       False  False


# Step 4 - Apropri: Find the frequent itemsets

In [89]:
''' 
apriori() keeps only the combinations of products that appear often enough. 
    - basket_table: the true/false table from Step 3
    - min_support=0.02: Keep an itemset only if it appears in at least 2% of the baskets (transactions)
                          Support is a fraction: 0.02 * 2472 baskets = 49.44 baskets, so we keep itemsets that appear in at least 50 baskets
                          Higher value -> fewer, stronger itemssets and a faster run time
    - use_colnames=True: use the product names (column names) instead of column numbers in the output
                         if false, it will not show the product names, only the column numbers (0, 1, 2, ...) which is not useful
    - max_len=3: Keep only itemsets with 3 or fewer products (1, 2, or 3 products). 
                 Longer ones are raise and hard to act on. Set it to none for no limit
'''

frequent_itemsets = apriori(basket_table, 
                         min_support=0.02, 
                         use_colnames=True, 
                         max_len=3) 

# At this point, the result has 2 columns: 'support' (the fractions) and 'itemsets' (the products in the itemset)

# Add a helper column: how many products are in each itemset
frequent_itemsets['size'] = frequent_itemsets['itemsets'].apply(len) # apply(len) = length of each set

print(frequent_itemsets)

print('\nSTEP 4 - frequent itemsets found by Apriori.')
print(f'  total itemsets: {len(frequent_itemsets)}')
by_size = frequent_itemsets['size'].value_counts().sort_index() # count how many itemsets of each size (1, 2, 3) and sort by size
print(f'  by size: {by_size.to_dict()}') # print the counts as a dictionary
print('  the 5 most frequent items of 2 products: ')
pairs = frequent_itemsets[frequent_itemsets['size']==2].nlargest(5, 'support') # get the 5 pairs with the largest support
# same as: .sort_values('support', ascending=False).head(5)
for i, row in pairs.iterrows():
    print(f'        {row["itemsets"]} : {row["support"]:.3f}') # print the itemset and its support

trash = frequent_itemsets
trash['readable'] = list(trash['itemsets'].apply(lambda x: ', '.join(x))) # convert the set of products into a readable string
trash

      support                                       itemsets  size
0    0.050162                            frozenset({apples})     1
1    0.096683                           frozenset({avocado})     1
2    0.082524                      frozenset({baby formula})     1
3    0.092233                        frozenset({baby wipes})     1
4    0.057848                             frozenset({bacon})     1
..        ...                                            ...   ...
179  0.030340         frozenset({pasta, red wine, parmesan})     3
180  0.062298     frozenset({pasta, parmesan, tomato sauce})     3
181  0.024272  frozenset({red wine, parmesan, tomato sauce})     3
182  0.033981     frozenset({pasta, red wine, tomato sauce})     3
183  0.023867    frozenset({peanuts, tortilla chips, salsa})     3

[184 rows x 3 columns]

STEP 4 - frequent itemsets found by Apriori.
  total itemsets: 184
  by size: {1: 50, 2: 86, 3: 48}
  the 5 most frequent items of 2 products: 
        frozenset({'pasta',

,support,itemsets,size,readable
0,0.050162,frozenset({apples}),1,apples
1,0.096683,frozenset({avocado}),1,avocado
2,0.082524,frozenset({baby formula}),1,baby formula
3,0.092233,frozenset({baby wipes}),1,baby wipes
4,0.057848,frozenset({bacon}),1,bacon
...,...,...,...,...
179,0.030340,"frozenset({pasta, red wine, parmesan})",3,"pasta, red wine, parmesan"
180,0.062298,"frozenset({pasta, parmesan, tomato sauce})",3,"pasta, parmesan, tomato sauce"
181,0.024272,"frozenset({red wine, parmesan, tomato sauce})",3,"red wine, parmesan, tomato sauce"
182,0.033981,"frozenset({pasta, red wine, tomato sauce})",3,"pasta, red wine, tomato sauce"


# Step 5 - Generate the rules

In [90]:
'''
association_rules() splits every frequent itemset into IF -> THEN rules, and calculates the metrics (quality measures) for each rule.
    - frequent_itemsets: the frequent itemsets from Step 4
    - metric='lift': the metric to use for filtering the rules (lift = 1 means no association, lift > 1 means positive association)
    - min_threshold=1.2: keep only rules with lift >= 1.2
        - lift = confidence / support(Y)
               = 1 -> useless
               > 1 -> positive association (if X is bought, Y is more likely to be bought)
               < 1 -> negative association (if X is bought, Y is less likely to be bought)


'''

rules = association_rules(frequent_itemsets,
                          metric='lift', # the metric to use for filtering the rules
                          min_threshold=1.2) # keep only rules with lift >= 1.2 (lift = 1 means no association, lift > 1 means positive association)
                          
# Keep only the rules we can actually serve
 
# 1) ONE product on the THE side: we recommend single items, not bundles
'''
rules['support'] in the market basket analysis, "support" is the percentage of all transactions 
that contain the items in the rule (the IF and THEN side together), e.g. 0.02 means 2% of all baskets

len(basksets) is the total number of baskets (transactions) in the dataset, e.g. 2472

rules['support'] * len(baskets) is the number of baskets that contain the items in the rule 
    (the IF and THEN side together), e.g. 0.02 * 2472 = 49.44 baskets, so we keep itemsets that appear in at least 50 baskets

.round().astype(int) cleans up the math to turn it into a whole number

.apply(len) == 1: keeps only rules with one product on the THEN side (the consequents)

rules['confidence'] >= 0.30: at least 30% of time when the IF side is bought, the THEN side is also bought

at least 40 real baskets behind the rule, so it is not a coincidence: 
    we want to be able to actually serve the recommendation, so we need at least 40 baskets that contain the rule (the IF and THEN side together)
    (support is a fraction, so support x number of baskets = number of baskets that contain the rule)
'''
rules['n_baskets'] = (rules['support'] * len(baskets)).round().astype(int) # portion of baskets that contain the rule (support * total number of baskets)

# print(rules) # use this if I want to see all the rules before the filtering
# print(len (baskets))

rules = rules[(rules['consequents'].apply(len) == 1) &
              (rules['confidence'] >= 0.30) & 
              (rules['n_baskets'] >= 40)] # see above for explanation of the filtering criteria
rules

# print(len (baskets))

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,n_baskets
1,frozenset({avocado}),frozenset({bananas}),0.096683,0.110437,0.029531,0.305439,2.765736,1.0,0.018853,1.280757,0.706765,0.166287,0.219211,0.286419,73
2,frozenset({avocado}),frozenset({berries}),0.096683,0.134304,0.036003,0.372385,2.772697,1.0,0.023018,1.379342,0.707770,0.184647,0.275017,0.320229,89
5,frozenset({avocado}),frozenset({granola}),0.096683,0.125405,0.030744,0.317992,2.535727,1.0,0.018620,1.282382,0.670457,0.160677,0.220201,0.281576,76
7,frozenset({avocado}),frozenset({yogurt}),0.096683,0.141990,0.039239,0.405858,2.858349,1.0,0.025511,1.444115,0.719734,0.196755,0.307534,0.341106,97
8,frozenset({baby formula}),frozenset({baby wipes}),0.082524,0.092233,0.028317,0.343137,3.720330,1.0,0.020706,1.381974,0.796977,0.193370,0.276397,0.325077,70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
447,"frozenset({pasta, tomato sauce})",frozenset({red wine}),0.104369,0.125405,0.033981,0.325581,2.596249,1.0,0.020892,1.296814,0.686476,0.173554,0.228879,0.298275,84
448,"frozenset({red wine, tomato sauce})",frozenset({pasta}),0.042071,0.173948,0.033981,0.807692,4.643292,1.0,0.026662,4.295469,0.819096,0.186667,0.767197,0.501521,84
452,"frozenset({peanuts, tortilla chips})",frozenset({salsa}),0.039644,0.104773,0.023867,0.602041,5.746119,1.0,0.019714,2.249544,0.860066,0.197987,0.555465,0.414920,59
453,"frozenset({peanuts, salsa})",frozenset({tortilla chips}),0.034385,0.122977,0.023867,0.694118,5.644272,1.0,0.019639,2.867189,0.852130,0.178788,0.651226,0.444098,59


In [91]:
# Sort the best rules first, i.e. highest lift first (the higher the lift, the stronger the association)
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True) # reset the index to be 0, 1, 2, ... instead of the original index from the dataframe
rules
# ic(rules)

print('\nSTEP 5 - association rules generated.')
print(f'  Usable rules: {len(rules)}')
print('  The 5 best rules (highest lift): ')
print(rules.head(5)[['antecedents', 'consequents', 'support', 'confidence', 'lift', 'n_baskets']]) # print the first 5 rules with the relevant columns since the rules have been sorted

print("\nAnother way to print the rules in a more readable format:")
for _, r in rules.head(5).iterrows():
    print(f'        {r["antecedents"]} -> {r["consequents"]} : support={r["support"]:.3f}, confidence={r["confidence"]:.3f}, lift={r["lift"]:.3f}, n_baskets={r["n_baskets"]}') # print the rule and its metrics
    
print("\nAnother way to print the rules without the word frozenset:")
for _, r in rules.head(5).iterrows():
    left = ', '.join(r["antecedents"]) # frozenset -> readable text
    right = ', '.join(r["consequents"]) 
    print(f"    IF {left:<28} | THEN {right:<16} | support={r['support']:.3f}, confidence={r['confidence']:.3f}, lift={r['lift']:.3f}, n_baskets={r['n_baskets']}") # print the rule and its metrics


STEP 5 - association rules generated.
  Usable rules: 245
  The 5 best rules (highest lift): 
                             antecedents                  consequents  \
0  frozenset({baby formula, baby wipes})         frozenset({diapers})   
1     frozenset({baby formula, diapers})      frozenset({baby wipes})   
2       frozenset({diapers, baby wipes})    frozenset({baby formula})   
3       frozenset({chocolate bar, beer})  frozenset({tortilla chips})   
4               frozenset({beer, salsa})  frozenset({tortilla chips})   

    support  confidence      lift  n_baskets  
0  0.023463    0.828571  7.038586         58  
1  0.023463    0.637363  6.910353         58  
2  0.023463    0.513274  6.219677         58  
3  0.023463    0.743590  6.046559         58  
4  0.039239    0.723881  5.886292         97  

Another way to print the rules in a more readable format:
        frozenset({'baby formula', 'baby wipes'}) -> frozenset({'diapers'}) : support=0.023, confidence=0.829, lift=7.039, n_

In [92]:
from icecream import ic
a, b = 5, 19
print (a, b)
ic(a, b)

ic| a: 5, b: 19


5 19


(5, 19)

# Step 6 - Save the rules to a file

In [93]:
'''
The rules are the "knowledge" of the recommender. We compute them once (slow)
and save them, so the recommendation part can just read the file.

A frozenset cannot be saved to a csv file, so we convert both sides to the text
'''

rules_to_save = rules.copy() # make a copy of the rules dataframe to save to csv
# convert the frozenset to a string. In lambda x is the frozenset, ', '.join(x) converts 
# it to a string with the items separated by commas, which are sorted 
rules_to_save['antecedents'] = rules_to_save['antecedents'].apply(lambda x: ', '.join(sorted(x))) 
rules_to_save['consequents'] = rules_to_save['consequents'].apply(lambda x: ', '.join(sorted(x)))
rules_to_save

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,n_baskets
0,"baby formula, baby wipes",diapers,0.028317,0.117718,0.023463,0.828571,7.038586,1.0,0.020129,5.146642,0.882928,0.191419,0.805699,0.513942,58
1,"baby formula, diapers",baby wipes,0.036812,0.092233,0.023463,0.637363,6.910353,1.0,0.020067,2.503236,0.887978,0.222222,0.600517,0.445874,58
2,"baby wipes, diapers",baby formula,0.045712,0.082524,0.023463,0.513274,6.219677,1.0,0.019690,1.884996,0.879420,0.223938,0.469495,0.398794,58
3,"beer, chocolate bar",tortilla chips,0.031553,0.122977,0.023463,0.743590,6.046559,1.0,0.019582,3.420388,0.861810,0.179012,0.707636,0.467190,58
4,"beer, salsa",tortilla chips,0.054207,0.122977,0.039239,0.723881,5.886292,1.0,0.032573,3.176244,0.877691,0.284457,0.685163,0.521480,97
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
240,croissant,milk 2%,0.099110,0.181230,0.033576,0.338776,1.869315,1.0,0.015614,1.238264,0.516206,0.136066,0.192418,0.262022,83
241,butter,coffee beans,0.167476,0.212783,0.063916,0.381643,1.793575,1.0,0.028280,1.273077,0.531461,0.202046,0.214502,0.341011,158
242,coffee beans,butter,0.212783,0.167476,0.063916,0.300380,1.793575,1.0,0.028280,1.189967,0.562049,0.202046,0.159640,0.341011,158
243,strawberry jam,coffee beans,0.138754,0.212783,0.049757,0.358601,1.685286,1.0,0.020233,1.227343,0.472140,0.164879,0.185232,0.296220,123


In [94]:
# Keep only useful columns, in a sensible order for reading
rules_to_save = rules_to_save[['antecedents', 'consequents', 'support', 'confidence', 'lift', 'n_baskets']]
rules_to_save.to_csv('association_rules.csv', index=False) # save the rules to a csv file

print('\nSTEP 6 - association rules saved to file.')
rules_to_save


STEP 6 - association rules saved to file.


,antecedents,consequents,support,confidence,lift,n_baskets
0,"baby formula, baby wipes",diapers,0.023463,0.828571,7.038586,58
1,"baby formula, diapers",baby wipes,0.023463,0.637363,6.910353,58
2,"baby wipes, diapers",baby formula,0.023463,0.513274,6.219677,58
3,"beer, chocolate bar",tortilla chips,0.023463,0.743590,6.046559,58
4,"beer, salsa",tortilla chips,0.039239,0.723881,5.886292,97
...,...,...,...,...,...,...
240,croissant,milk 2%,0.033576,0.338776,1.869315,83
241,butter,coffee beans,0.063916,0.381643,1.793575,158
242,coffee beans,butter,0.063916,0.300380,1.793575,158
243,strawberry jam,coffee beans,0.049757,0.358601,1.685286,123


# Step 7 - The Recommender

In [95]:
'''
Suggest products for a basket, with an explanation of why we suggest them.

Recommend products based on the customer's existing basket and the association rules learned from historial shopping transactions.

Parameters: 
- basket: a list of products in the customer's basket
- how many: how many suggestions to return (default 3). Maximum number of recommendations of products to return

Returns: 
- a list of dictionaries, each dictionary contains: {'item', 'score', 'reason'}
    - item: the product to recommend
    - score: list of score rounded to 2 decimal places
    - reason: explanation for the recommendation, i.e. the rule that led to the recommendation, in a readable format
    
Requirements: 
The dataframe of rules or Json file must always exist.

'''

# TODO Load rules from a file, csv or json, so we don't have to recompute them every time we run the program.
# Create a streamlit app that allows the user to input a basket (by keyboard or a list to choose) and get recommendations based on the rules.

def recommend(basket, how_many=3): 
    # Step 7.1 Convert the basket into a set
    '''
    Example: 
    ['pasta', 'ground beef', 'pasta'] -> {'pasta', 'ground beef'}
    Sets remoe duplicates and make membership checks convenient
        
    '''
    basket = set(basket) # convert the basket into a set to remove duplicates and make membership checks convenient  
                         # a set makes the "is it inside?" test easy  
    
    # Step 7.2 Create dictionnaries to store recommendations
    '''
    Dictionarly storing each recommended product and its best life score.add()
    {
        'tomato source': 2.4
        'parmesan': 1.8
    }
    '''
    
    suggestions = {} # product -> best lift score so format
    
    '''
    Dicionary storing the explanation assocaited with each product
    
    The product name is the key, and its explanataion is the value
    
    Example: 
    {
        'tomato sauce': 'because you bought pasta ...'
    }
    '''
    
    reason = {}
    
    # Step 7.3 Examine every association rule
    '''
    'rules' is a panda datafram that contains the association_rules learned in the previous steps
    '.iterrows()' returns an iterator over the rows of the dataframe, each row is a tuple (index, row)
    '_' receives the row index, which we don't need, and 'row' receives the row itself, which is a pandas Series object
    'rule' contains the current row's data
    '''
    
    # Look at every rule we kept in Step 5 (the rules that are strong enough to be useful)
    for _, rule in rules.iterrows(): 
        # Step 7.3.1 Get the antecedent (the IF side)
        '''
        Retrieve the products requried for this rule to apply
        Example: 
        {'pasta', 'ground beef'}
        
        Convert them into a set for easy comparison with the basket
        '''
        
        if_part = set(rule['antecedents'])
        
        # Step 7.3.2 Get the consequent (the THEN side)
        '''
        Retrieve the product that is recommended by this rule
        'list()' converts the consequent collection into a list
        '[0]' selects the first product in the list
        
        '''
        
        # get the first (and only) item in the set of consequents, since we only kept rules with one consequent
        then_part = list(rule['consequents'])[0]
        # print(if_part, then_part)
        
        if if_part.issubset(basket):
            
            # Step 7.4 Check whether the rule applies to the basket
            '''
            '.issubset()' checks whether the antecedent (the IF side) is a subset of the basket
            'not' reverses the result
            If even one required product is missing, skip this rule and move on to the next one
            because the if part must be fully contained in the basket for the rule to apply
            '''
            
            if not if_part.issubset(basket):
                continue

            # Step 7.5 Check whether the recommended product is already in the basket
            '''
            Check whether the recommended product (the THEN side) is already in the basket
            If it is, skip this rule and move on to the next one            
            because we don't want to recommend products that the customer already has
            '''
            
            if then_part in basket:
                continue
            
            # Step 7.6 Keep the strongest rule for each recommended product
            '''
            A product may be recommended by multiple rules, but we want to keep only the strongest one (the one with the highest lift score)
            
            Example: 
            pasta               -> tomato sauce (lift=1.7)
            pasta + ground beef -> tomato sauce (lift=1.8)
            
            In this case, we want to keep the second rule, because it has a higher lift score (1.8 > 1.7)
            
            'or' means the condition is true if the product is New or 
            the current rule has a higher lift score than the previous one (the one already stored in the suggestions dictionary)
            
            
            
            If the product is not already in the suggestions dictionary, add it with its lift score and explanation
            If the product is already in the suggestions dictionary, but this rule has a higher lift score than the previous one, update it with the new lift score and explanation
            Store the lift score for this product in the suggestions dictionary, and store the explanation for this product in the reason dictionary
            
            '''
            if ((then_part not in suggestions) # if the product is not already in the suggestions dictionary, 
                                              # add it with its lift score and explanation
                or (rule['lift'] > suggestions[then_part]) # if the product is already in the suggestions dictionary, 
                ):                                         # but this rule has a higher lift score than the previous one, 
                                                           # update it with the new lift score and explanation
                
                # store the strongest lift score for this product                
                suggestions[then_part] = rule['lift'] 
                
                # Sort the strongest lift score found for this product
                '''
                Store the antecedent products alphabetically and join them into a readable phrase
                
                Example: 
                {'pasta', 'ground beef'} -> 'ground beef, pasta'
               
                '''
                
                trigger = ' and '.join(sorted(if_part)) # sort the antecedent products alphabetically and join them into a readable phrase
                
                # Build an explanation for this product using f-string
                '''
                {trigger} inserts the products that activated the rule (the IF side) into the explanation
                {then_part} inserts the recommended product (the THEN side) into the explanation
                {rule['confidence']:.0%} inserts the confidence score of the rule into the explanation
                {rule['lift']:.2f} inserts the lift score of the rule into the explanation               
                '''
                reason[then_part] = (
                    f"because you bought {trigger} - "
                    f"(confidence={rule['confidence']:.0%} of baskets with"
                    f"{trigger} also contain {then_part}, "
                    f"lift={rule['lift']:.2f})x more often than average "
                    f"(based on {rule['n_baskets']} real baskets)"
                )
                
                # Save the explanation udner the recommended product's name
                # if a stronger rule is found later, it will overwrite the previous explanation with the new one      

    # Step 7.67 Sort the recommendations from the highest score to the lowest score, keep the top ones
    # sorted(..., key = ..., reverse=True) sorts the items in the suggestions dictionary by their lift score in descending order
    best = sorted(suggestions.items(), 
                    key=lambda pair: pair[1], # lambda pair: pair[1] means sort by the second item in the tuple (the lift score)
                    reverse=True)[:how_many] # keep only the top 'how_many' recommendations
    
    '''
    Example of the output format:
    [
        {
            'item': 'tomato sauce', # the recommended product
            'score': 2.4, # the lift score rounded to 2 decimal places
            'reason': "because you bought tomato sauce ..." # the explanation for the recommendation
        }
    ]
    If no eligible recommendations are found, the output will be an empty list: []
    
    '''
        
    return [
        {
            'item': item, # the recommended product
            'score': round(score, 2), # the lift score rounded to 2 decimal places
            'reason': reason[item] # the explanation for the recommendation
        }
        for item, score in best # for each recommended product and its lift score in the top recommendations
    ]

# Step 8 - Test the recommend function with a sample basket_table

In [96]:
print("\nSTEP 8 - test the recommend function with a sample basket")

# Note we may need to parse the basket items to match the cleaned product names in the rules (lowercase, no leading/trailing spaces)
test_baskets = [
    ['pasta'],                         
    ['pasta', 'ground beef'], 
    ['beer', 'tortilla chips'],               
    ['diapers'],                 
    ['yogurt', 'berries'],             
    ['shampoo'],
    ['lemon'],
    ['bananas', 'berries']
]

for id, basket in enumerate(test_baskets):
    print(f"\nBasket: {id+1} - {basket}")
    recommendations = recommend(basket, how_many=3)
    if recommendations:
        for rec in recommendations:
            print(f"  Recommend: {rec['item']} (lift={rec['score']}) - {rec['reason']}")
    else:
        print("  No recommendations found (no rules apply to this basket).")



STEP 8 - test the recommend function with a sample basket

Basket: 1 - ['pasta']
  Recommend: tomato sauce (lift=3.67) - because you bought pasta - (confidence=60% of baskets withpasta also contain tomato sauce, lift=3.67)x more often than average (based on 258 real baskets)
  Recommend: parmesan (lift=3.39) - because you bought pasta - (confidence=47% of baskets withpasta also contain parmesan, lift=3.39)x more often than average (based on 202 real baskets)

Basket: 2 - ['pasta', 'ground beef']
  Recommend: parmesan (lift=4.49) - because you bought ground beef and pasta - (confidence=62% of baskets withground beef and pasta also contain parmesan, lift=4.49)x more often than average (based on 71 real baskets)
  Recommend: tomato sauce (lift=4.4) - because you bought ground beef and pasta - (confidence=72% of baskets withground beef and pasta also contain tomato sauce, lift=4.40)x more often than average (based on 82 real baskets)

Basket: 3 - ['beer', 'tortilla chips']
  Recommend: sa